## WEEK 6 へようこそ

壮大な最終週です

そして

# **M**ODEL **C**ONTEXT **P**ROTOCOL へようこそ!

そして OpenAI Agents SDK へも、ようこそ戻ってきました。

In [ ]:
# インポート

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import Image, display
import os

In [ ]:
load_dotenv(override=True)

In [ ]:
# メインモデルをOpenAIからGeminiに切り替える
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=GEMINI_BASE_URL)
MODEL_NAME = OpenAIChatCompletionsModel(model="gemini-flash-latest", openai_client=gemini_client)

# GeminiのキーではOpenAIのトレース機能(platform.openai.com/traces)は使えないため無効化
set_tracing_disabled(True)


### OpenAI Agents SDK で MCP を使ってみよう

MCP を使うと、エージェントは他の誰かが書き、別のプログラムとして動くツールを使えるようになります。まず小さなクライアントを起動すると、それがサーバーをサブプロセスとして立ち上げ、サーバーは自分が提供するツールを報告してきます。ひとつのサーバーのツール一覧を取得できるようになれば、他のどのサーバーでも同じ方法で使えます。

ここでは3つのローカルサーバーに出会い、そのうちいくつかをエージェントに渡します。最初は Fetch サーバーです。これは Web ページを取得し、綺麗な markdown として返してくれます。

`async with` はサーバーを起動し、ブロックが終わるとシャットダウンします。これらは `uvx` や `npx` を通じて実行され、最初は遅くなることがあるので、デフォルトの短いタイムアウトではなく、余裕を持ったタイムアウトをクライアントに設定します。

### Windows での注意点

ノートブックからローカルの MCP サーバーを起動すると、Windows では厄介な問題にぶつかります。サーバーは起動時の出力を stderr に書き込みますが、Windows の Jupyter カーネルの中ではこのストリームの裏に実際のファイルハンドルがないため、起動が `io.UnsupportedOperation: fileno` エラーで失敗します。一方、Mac と Linux では影響がありません。

対処法は、サーバーの stderr をヌルデバイスに送ることです。これにより、サーバーは常に書き込める実在の場所を持つことになります。次のセルでこれを一度だけ行えば、以降のすべてのセルで、OpenAI Agents SDK のドキュメントどおりに `MCPServerStdio` をそのまま使えるようになります。Mac と Linux では、サーバーの起動時バナーがノートブックに表示されなくなるだけで、他に影響はありません。

In [ ]:
# Windows では、Jupyter カーネルから起動した stdio MCP サーバーが、実際のファイルディスクリプタを
# 持たない stderr ストリームに書き込もうとして io.UnsupportedOperation: fileno でクラッシュする。
# サーバーの stderr をヌルデバイスに送ることで、常に書き込める実在の場所を用意し、これにより
# 以降のすべてのセルで OpenAI Agents SDK のドキュメントどおりに MCPServerStdio を使えるようにする。
# Mac と Linux では影響がない。
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [ ]:
fetch_params = {"command": "uvx", "args": ["--with", "mcp<2","mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

In [ ]:
print(fetch_tools[0].description)

In [ ]:
fetch_tools[0].inputSchema

## Node と Playwright

ここまでは `uvx` を通じて Python の MCP サーバーを使ってきましたが、次の2つは JavaScript のランタイムである Node 上で、`npx` を通じて動きます。始める前に、簡単な確認を2つしておきましょう。

まず Node 自体です。v22 以降が必要です。下のセルが失敗する場合は、次のコマンドのいずれかで Node をインストールしてください。

- **Windows** の場合、PowerShell で: `winget install OpenJS.NodeJS.LTS`
- **Mac** の場合、Terminal で: `brew install node`
- **Linux**、またはどちらもうまくいかない場合: [setup/SETUP-node.md](../setup/SETUP-node.md) のガイドを参照してください

インストール後は、Cursor を完全に終了してから再度起動し、このノートブックを開き直して、ラボを最初からやり直してください。この完全な再起動が重要です。新しくインストールされた Node は、すでに実行中のノートブックからは見えません。カーネルだけを再起動しても不十分です。

In [ ]:
!node --version
!npx --version

次に、Microsoft のブラウザ自動化フレームワークである Playwright です。インストールするものは何もありません。`npx` が必要に応じて取得してくれ、あなたのマシンにすでにある Chrome を操作します。Chrome が起動している必要もありません。Playwright が自分で立ち上げます。

下のセルは、AI を一切使わずにこの一連の流れ全体を証明します。Node が Playwright を実行し、Playwright が Chrome を開いて Hacker News を読み込み、スクリーンショットを保存します。npx がパッケージをダウンロードするため、最初の実行は少し時間がかかります。

Chrome が見つからないと言われた場合は、Chrome を普通にインストールするか、ターミナルで `npx playwright install chrome` を実行してから、もう一度試してください。

In [ ]:
!npx -y playwright@latest screenshot --channel=chrome https://news.ycombinator.com playwright_check.png
display(Image("playwright_check.png"))

## さらに2つの MCP サーバー - 今度は JavaScript ベース、node を使う

Playwright は本物の Web ブラウザを操作するので、エージェントはページを開いてクリックして回ることができます。filesystem サーバーはファイルの読み書きを行いますが、ここでは単一の `sandbox` フォルダを指定するので、エージェントはそのフォルダ内のファイルにしか触れず、あなたのマシンの他の部分には一切触れません。

### 次のセルがハングする場合は、上でリンクした SETUP-node の手順の末尾にある重要なトラブルシューティングを参照してください。

### Windows で次のセルが失敗する場合、コマンドの実行ファイルのパスにスペースが含まれていることが原因かもしれません。

* 例: `C:\Program Files\nodejs\npx.ps1`

ここではフルパスを書いていませんが、`MCP` ライブラリが裏でそれを展開します。

もっとも簡単な修正方法は、パスが安全なネイティブのシェルインタープリタ、`powershell` や `cmd` を経由してコマンドを実行することです。

* 次を置き換えます:

  ```python
  playwright_params = {"command": "npx", "args": ["@playwright/mcp@latest"]}
  ```

* 次のように:

  ```python
  playwright_params = {"command": "powershell", "args": ["/c", "npx", "@playwright/mcp@latest"]}
  ```

これでうまくいったら、`MCPServerStdioParams` オブジェクトを組み立てるときに `command` のパスにスペースが含まれている場合は、常に同じパターンを使うことを覚えておいてください。

* `command` を `powershell` または `cmd` に置き換え、元のコマンドとその引数を `"/c"` を先頭に付けて `args` に入れます。

In [ ]:

playwright_params = {"command": "npx", "args": [ "@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    playwright_tools = await server.list_tools()

playwright_tools


In [ ]:


# sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
# os.makedirs(sandbox_path, exist_ok=True)

# sandbox_path

In [ ]:

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))  # filesystem サーバーが触れてよい唯一のフォルダ
os.makedirs(sandbox_path, exist_ok=True)
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

file_tools

### さあ、ツールを持ったエージェントの登場です!

2つのサーバーを1つのエージェントに渡し、ゴールを与えます。エージェントが自分で、いつ Playwright でブラウジングし、いつ filesystem サーバーでファイルを書くかを決める様子を観察し、それからトレースを開いて、エージェントが行ったツール呼び出しを確認してみましょう。

In [ ]:
INSTRUCTIONS = """
You browse the internet to accomplish your instructions.
Accept cookies and navigate pop-ups as needed.
If one website isn't fruitful, try another.
Be persistent until you have solved your assignment,
trying different options and sites as needed.
When you need to write files, you do that inside the sandbox/ folder only.
"""

TASK = "Find a great recipe for Banoffee Pie, then summarize it in markdown to banoffee.md"


async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
        agent = Agent(
            name="investigator",
            instructions=INSTRUCTIONS,
            model=MODEL_NAME,
            mcp_servers=[mcp_server_files, mcp_server_browser]
            )
        with trace("investigate"):
            result = await Runner.run(agent, TASK, max_turns=20)
            print(result.final_output)

### もう一つ: リモートの MCP サーバー

ここまでのサーバーはすべてローカルでした。`MCPServerStdio` にコマンドを渡すと、それがあなたのマシン上でサーバーをサブプロセスとして立ち上げ、stdin と stdout でやり取りしていました。しかし多くの MCP サーバーは、代わりに HTTP 経由でアクセスするホスト型サービスとして動きます。そうしたサーバーには `MCPServerStreamableHttp` と URL で接続でき、何も起動する必要もインストールする必要もありません。

これは Week 2 で少し触れた Context7 の話です。Context7 は、ライブラリの最新のドキュメントを調べてくれるホスト型サーバーで、エージェントに学習データより新しい事実を渡す、なかなか便利な方法です。まず何の助けもなく質問をし、次に Context7 をエージェントに与えてもう一度質問してみます。

In [ ]:
from agents.mcp import MCPServerStreamableHttp

In [ ]:
question = """
In the SandboxAgents feature added to the OpenAI Agents SDK in 2026, what is the Manifest object for?
Be accurate. If you don't know the answer, don't guess. State that you don't know.
"""

In [ ]:
agent = Agent(name="Expert", instructions="Answer the question.", model=MODEL_NAME)
result = await Runner.run(agent, question)
print(result.final_output)

In [ ]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}

async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question.", mcp_servers=[server], model=MODEL_NAME)
    result = await Runner.run(agent, question)

print(result.final_output)

### トレースを確認してみましょう

https://platform.openai.com/traces

### そして、いくつかの MCP マーケットプレイスも見てみましょう

https://glama.ai/mcp

https://smithery.ai/
